# Results Summary: All Models Comparison

**Purpose**: Consolidate and compare results from all classification approaches, including the previously missing Jensen-Shannon Divergence (JSD) metric.

**Models compared**:
1. TF-IDF + Logistic Regression (baseline)
2. TF-IDF + Linear SVC (baseline)
3. KB-BERT Multi-label Binary Classification
4. KB-BERT Direct Distributional Prediction

**Metrics**:
- Binary: Subset Accuracy, Micro-F1, Macro-F1, Hamming Loss
- Distributional: MAE, Top-1 Accuracy, Cosine Similarity, **Jensen-Shannon Divergence**

In [ ]:
# Imports
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy.spatial.distance import cosine, jensenshannon
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.precision', 4)

print('Imports OK')

## 1. Configuration

Adjust paths based on your Kaggle setup. The notebook expects:
- Preprocessed data exports from the baseline notebook
- Saved baseline model pipelines
- BERT prediction outputs

In [ ]:
# === ADJUST PATHS FOR YOUR KAGGLE SETUP ===

# Option A: If running after baseline notebook in same session
EXPORTS_DIR = Path("/kaggle/working/exports")
BASELINE_MODEL_DIR = Path("/kaggle/working/tfidf_baseline")

# Option B: If loading from saved Kaggle datasets
# EXPORTS_DIR = Path("/kaggle/input/prepreprocessed-for-bert")
# BASELINE_MODEL_DIR = Path("/kaggle/input/tfidf-baseline-models")

# BERT predictions (adjust to your saved output location)
BERT_BINARY_PREDS = Path("/kaggle/working/bert_expert_review_cases.csv")  # or your saved path
BERT_DIST_PREDS = Path("/kaggle/working/bert_distributional_predictions.csv")

print("Paths configured")

## 2. Helper Functions

In [ ]:
def deserialize_tuple(s):
    """Convert JSON string back to tuple."""
    import json
    if pd.isna(s) or s == "":
        return ()
    try:
        return tuple(json.loads(s))
    except:
        return ()


def make_gold_distribution(row, uo_to_idx, num_labels):
    """
    Convert (labels_uo, labels_pct) tuples into a distribution vector.
    """
    dist = np.zeros(num_labels, dtype=float)
    uos = row["labels_uo"]
    pcts = row["labels_pct"]
    
    if not isinstance(uos, (list, tuple)) or len(uos) == 0:
        return dist
    
    if not isinstance(pcts, (list, tuple)) or len(pcts) != len(uos):
        equal_pct = 100.0 / len(uos)
        pcts = [equal_pct] * len(uos)
    
    for uo, pct in zip(uos, pcts):
        if uo in uo_to_idx:
            dist[uo_to_idx[uo]] = pct
    
    return dist


def normalize_to_distribution(probs, scale=100):
    """
    Convert probabilities to normalized distribution summing to scale.
    """
    probs = np.maximum(probs, 0)
    row_sums = probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    return (probs / row_sums) * scale


def compute_all_distributional_metrics(gold_dist, pred_dist, scale=100):
    """
    Compute ALL distributional metrics including JSD.
    
    Args:
        gold_dist: Ground truth distributions (n_samples, n_labels)
        pred_dist: Predicted distributions (n_samples, n_labels)
        scale: Sum of distributions (100 for percentages, 1 for probabilities)
    
    Returns:
        dict with all metrics
    """
    # Normalize to probabilities for JSD
    gold_prob = gold_dist / scale if scale != 1 else gold_dist
    pred_prob = pred_dist / scale if scale != 1 else pred_dist
    
    # MAE in percentage points
    mae = np.abs(gold_dist - pred_dist).mean() * (100 / scale)
    
    # Top-1 Accuracy (primary label match)
    gold_top1 = np.argmax(gold_dist, axis=1)
    pred_top1 = np.argmax(pred_dist, axis=1)
    top1_acc = (gold_top1 == pred_top1).mean()
    
    # Cosine Similarity
    cosine_sims = []
    for g, p in zip(gold_dist, pred_dist):
        if g.sum() > 0 and p.sum() > 0:
            cosine_sims.append(1 - cosine(g, p))
        else:
            cosine_sims.append(0.0)
    
    # Jensen-Shannon Divergence
    js_divs = []
    for g, p in zip(gold_prob, pred_prob):
        # Add epsilon and renormalize for numerical stability
        g_safe = g + 1e-10
        p_safe = p + 1e-10
        g_safe = g_safe / g_safe.sum()
        p_safe = p_safe / p_safe.sum()
        js_divs.append(jensenshannon(g_safe, p_safe))
    
    return {
        "mae_pct": mae,
        "top1_accuracy": top1_acc,
        "mean_cosine_sim": np.mean(cosine_sims),
        "mean_jsd": np.mean(js_divs),
        "std_jsd": np.std(js_divs),
        "per_sample_mae": np.abs(gold_dist - pred_dist).mean(axis=1) * (100 / scale),
        "per_sample_jsd": np.array(js_divs),
    }


def compute_binary_metrics(y_true, y_pred):
    """
    Compute multi-label binary classification metrics.
    """
    return {
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "micro_f1": f1_score(y_true, y_pred, average='micro', zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


print("Helper functions defined")

## 3. Load Data

In [ ]:
# Load validation data and label list
val_df = pd.read_csv(EXPORTS_DIR / "val_ml_export.csv")
label_list = joblib.load(EXPORTS_DIR / "uo_label_list.joblib")
num_labels = len(label_list)

# Deserialize tuple columns
val_df["labels_uo"] = val_df["labels_uo"].apply(deserialize_tuple)
val_df["labels_pct"] = val_df["labels_pct"].apply(deserialize_tuple)

# Get binary ground truth
label_cols = [f"y_{lab}" for lab in label_list]
Y_val_binary = val_df[label_cols].values.astype(int)

# Build gold distribution
uo_to_idx = {code: i for i, code in enumerate(label_list)}
gold_dist = np.array([
    make_gold_distribution(row, uo_to_idx, num_labels)
    for _, row in val_df.iterrows()
])

print(f"Loaded {len(val_df)} validation samples")
print(f"Labels ({num_labels}): {label_list}")

## 4. Evaluate Baseline Models (TF-IDF)

In [ ]:
# Load saved baseline pipelines
logreg_pipe = joblib.load(BASELINE_MODEL_DIR / "logreg_pipe.joblib")
svc_pipe = joblib.load(BASELINE_MODEL_DIR / "svc_pipe.joblib")

# Generate predictions
Y_pred_logreg = logreg_pipe.predict(val_df["text"])
Y_pred_svc = svc_pipe.predict(val_df["text"])

# Binary metrics
logreg_binary = compute_binary_metrics(Y_val_binary, Y_pred_logreg)
svc_binary = compute_binary_metrics(Y_val_binary, Y_pred_svc)

# For distributional metrics: normalize binary predictions to distributions
logreg_dist = normalize_to_distribution(Y_pred_logreg.astype(float), scale=100)
svc_dist = normalize_to_distribution(Y_pred_svc.astype(float), scale=100)

logreg_dist_metrics = compute_all_distributional_metrics(gold_dist, logreg_dist)
svc_dist_metrics = compute_all_distributional_metrics(gold_dist, svc_dist)

print("Baseline models evaluated")

## 5. Evaluate BERT Binary Model

In [ ]:
# Option A: If you have saved the raw predictions
# Load BERT binary predictions (you may need to adjust based on what you saved)

# If predictions were saved as part of trainer.predict():
# bert_binary_preds = np.load("bert_binary_predictions.npy")

# Option B: Re-run prediction with saved model
# This requires the model to be loaded - see section below

# For now, let's set a flag for whether BERT predictions are available
BERT_BINARY_AVAILABLE = False  # Set to True if you have saved predictions

if BERT_BINARY_AVAILABLE:
    # Load your BERT binary predictions
    # Y_pred_bert_binary = ...
    # bert_probs = ...  # Sigmoid probabilities for distributional conversion
    
    bert_binary_metrics = compute_binary_metrics(Y_val_binary, Y_pred_bert_binary)
    bert_binary_dist = normalize_to_distribution(bert_probs, scale=100)
    bert_binary_dist_metrics = compute_all_distributional_metrics(gold_dist, bert_binary_dist)
    print("BERT Binary evaluated")
else:
    print("BERT Binary predictions not loaded - using values from notebook output")
    # Fallback: Use the values you already computed in the original notebook
    bert_binary_metrics = {
        "subset_accuracy": 0.9094,  # From your notebook
        "micro_f1": 0.9517,
        "macro_f1": 0.9260,
        "hamming_loss": 0.0112,
    }
    bert_binary_dist_metrics = {
        "mae_pct": 2.75,  # From your notebook
        "top1_accuracy": 0.851,
        "mean_cosine_sim": 0.94,
        "mean_jsd": None,  # NEEDS TO BE CALCULATED
    }

## 6. Evaluate BERT Distributional Model

In [ ]:
# Check if distributional predictions file exists
BERT_DIST_AVAILABLE = BERT_DIST_PREDS.exists()

if BERT_DIST_AVAILABLE:
    dist_results = pd.read_csv(BERT_DIST_PREDS)
    
    # Extract prediction columns
    pred_cols = [f"pred_{code}" for code in label_list]
    gold_cols = [f"gold_{code}" for code in label_list]
    
    bert_dist_pred = dist_results[pred_cols].values
    bert_dist_gold = dist_results[gold_cols].values
    
    # Compute all metrics including JSD
    bert_dist_metrics = compute_all_distributional_metrics(bert_dist_gold, bert_dist_pred)
    print("BERT Distributional evaluated from saved predictions")
else:
    print("BERT Distributional predictions not found - using values from notebook output")
    # Fallback values from your notebook
    bert_dist_metrics = {
        "mae_pct": 1.44,
        "top1_accuracy": 0.924,
        "mean_cosine_sim": 0.9511,
        "mean_jsd": None,  # NEEDS TO BE CALCULATED
    }

## 7. Results Summary Table

In [ ]:
# Build comparison table for BINARY metrics
binary_results = pd.DataFrame({
    "Model": ["TF-IDF + LogReg", "TF-IDF + LinearSVC", "KB-BERT Binary"],
    "Subset Acc.": [
        logreg_binary["subset_accuracy"],
        svc_binary["subset_accuracy"],
        bert_binary_metrics["subset_accuracy"],
    ],
    "Micro-F1": [
        logreg_binary["micro_f1"],
        svc_binary["micro_f1"],
        bert_binary_metrics["micro_f1"],
    ],
    "Macro-F1": [
        logreg_binary["macro_f1"],
        svc_binary["macro_f1"],
        bert_binary_metrics["macro_f1"],
    ],
    "Hamming Loss": [
        logreg_binary["hamming_loss"],
        svc_binary["hamming_loss"],
        bert_binary_metrics["hamming_loss"],
    ],
})

print("="*70)
print("BINARY CLASSIFICATION METRICS")
print("="*70)
print(binary_results.to_string(index=False))

In [ ]:
# Build comparison table for DISTRIBUTIONAL metrics
def safe_get(d, key, default="N/A"):
    val = d.get(key)
    return val if val is not None else default

dist_results_table = pd.DataFrame({
    "Model": [
        "TF-IDF + LogReg (normalized)",
        "TF-IDF + LinearSVC (normalized)",
        "KB-BERT Binary (normalized)",
        "KB-BERT Distributional",
    ],
    "MAE (pp)": [
        logreg_dist_metrics["mae_pct"],
        svc_dist_metrics["mae_pct"],
        safe_get(bert_binary_dist_metrics, "mae_pct"),
        safe_get(bert_dist_metrics, "mae_pct"),
    ],
    "Top-1 Acc.": [
        logreg_dist_metrics["top1_accuracy"],
        svc_dist_metrics["top1_accuracy"],
        safe_get(bert_binary_dist_metrics, "top1_accuracy"),
        safe_get(bert_dist_metrics, "top1_accuracy"),
    ],
    "Cosine Sim.": [
        logreg_dist_metrics["mean_cosine_sim"],
        svc_dist_metrics["mean_cosine_sim"],
        safe_get(bert_binary_dist_metrics, "mean_cosine_sim"),
        safe_get(bert_dist_metrics, "mean_cosine_sim"),
    ],
    "JSD": [
        logreg_dist_metrics["mean_jsd"],
        svc_dist_metrics["mean_jsd"],
        safe_get(bert_binary_dist_metrics, "mean_jsd"),
        safe_get(bert_dist_metrics, "mean_jsd"),
    ],
})

print("\n" + "="*70)
print("DISTRIBUTIONAL METRICS (including JSD)")
print("="*70)
print(dist_results_table.to_string(index=False))
print("\nNote: Lower JSD = better (0 = identical distributions)")
print("Note: pp = percentage points")

## 8. LaTeX-Ready Output

In [ ]:
def format_metric(val, fmt=".4f", multiply=1):
    """Format metric value for table."""
    if val is None or val == "N/A":
        return "--"
    return f"{val * multiply:{fmt}}"

print("\n" + "="*70)
print("LATEX TABLE - Binary Metrics")
print("="*70)
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"Model & Subset Acc. & Micro-F1 & Macro-F1 & Hamming Loss \\")
print(r"\midrule")

for _, row in binary_results.iterrows():
    print(f"{row['Model']} & {row['Subset Acc.']:.3f} & {row['Micro-F1']:.3f} & {row['Macro-F1']:.3f} & {row['Hamming Loss']:.4f} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{Multi-label binary classification results}")
print(r"\label{tab:binary-results}")
print(r"\end{table}")

In [ ]:
print("\n" + "="*70)
print("LATEX TABLE - Distributional Metrics")
print("="*70)
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"Model & MAE (pp) & Top-1 Acc. & Cosine Sim. & JSD \\")
print(r"\midrule")

for _, row in dist_results_table.iterrows():
    mae = format_metric(row['MAE (pp)'], ".2f")
    top1 = format_metric(row['Top-1 Acc.'], ".3f")
    cos = format_metric(row['Cosine Sim.'], ".4f")
    jsd = format_metric(row['JSD'], ".4f")
    print(f"{row['Model']} & {mae} & {top1} & {cos} & {jsd} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\caption{Distributional prediction results. Lower JSD indicates better distribution matching.}")
print(r"\label{tab:dist-results}")
print(r"\end{table}")

## 9. JSD Distribution Analysis

In [ ]:
# If we have per-sample JSD, analyze the distribution
if "per_sample_jsd" in bert_dist_metrics and bert_dist_metrics["per_sample_jsd"] is not None:
    jsd_samples = bert_dist_metrics["per_sample_jsd"]
    
    print("\n" + "="*70)
    print("JSD DISTRIBUTION ANALYSIS (BERT Distributional)")
    print("="*70)
    print(f"Mean JSD:   {np.mean(jsd_samples):.4f}")
    print(f"Median JSD: {np.median(jsd_samples):.4f}")
    print(f"Std JSD:    {np.std(jsd_samples):.4f}")
    print(f"Min JSD:    {np.min(jsd_samples):.4f}")
    print(f"Max JSD:    {np.max(jsd_samples):.4f}")
    
    # Percentiles
    print("\nPercentiles:")
    for p in [25, 50, 75, 90, 95, 99]:
        print(f"  {p}th: {np.percentile(jsd_samples, p):.4f}")
    
    # Interpretation guide
    print("\nInterpretation:")
    print("  JSD = 0.0: Identical distributions")
    print("  JSD < 0.1: Very similar distributions")
    print("  JSD < 0.2: Similar distributions")
    print("  JSD > 0.3: Notably different distributions")
else:
    print("Per-sample JSD not available - run with actual predictions to see distribution analysis")

## 10. Save Results

In [ ]:
# Save summary tables
binary_results.to_csv("/kaggle/working/results_binary_metrics.csv", index=False)
dist_results_table.to_csv("/kaggle/working/results_distributional_metrics.csv", index=False)

print("Results saved:")
print("  - results_binary_metrics.csv")
print("  - results_distributional_metrics.csv")